In [1]:
import os, sys

PROJECT_ROOT = r"C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from model import ConvNeXtV2TinyScratch

In [2]:
import os, json
import numpy as np
import pandas as pd

OUT_DIR = r"C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\ConvUnet\Baseline+ECA\Baseline+ECA(SamplerUNet++)GEMopt_Output"
KB_IDS = set()

need = [
    "checkpoint_infer.pt",
    "checkpoint_infer.json",
    "embeddings.npy",
    "rag_meta.csv",
]
for f in need:
    p = os.path.join(OUT_DIR, f)
    print(f, "OK" if os.path.exists(p) else "MISSING", p)

checkpoint_infer.pt OK C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\ConvUnet\Baseline+ECA\Baseline+ECA(SamplerUNet++)GEMopt_Output\checkpoint_infer.pt
checkpoint_infer.json OK C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\ConvUnet\Baseline+ECA\Baseline+ECA(SamplerUNet++)GEMopt_Output\checkpoint_infer.json
embeddings.npy OK C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\ConvUnet\Baseline+ECA\Baseline+ECA(SamplerUNet++)GEMopt_Output\embeddings.npy
rag_meta.csv OK C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\ConvUnet\Baseline+ECA\Baseline+ECA(SamplerUNet++)GEMopt_Output\rag_meta.csv


In [3]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd

# =========================
# 0) Load RAG library assets
# =========================
meta = pd.read_csv(os.path.join(OUT_DIR, "rag_meta.csv")).reset_index(drop=True)
embs = np.load(os.path.join(OUT_DIR, "embeddings.npy")).astype("float32")
assert len(meta) == embs.shape[0], f"meta rows={len(meta)} != embs rows={embs.shape[0]}"

def guess_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

PATH_COL   = guess_col(meta, ["full_path","image_path","img_path","path","filepath","file_path","png_path"])
CASEID_COL = guess_col(meta, ["image_id","ImageId","case_id","id","new_filename","filename","file_name"])
SPLIT_COL  = guess_col(meta, ["split","subset","set","stage","partition"])
YTRUE_COL  = guess_col(meta, ["y_true","label","gt","target","truth"]) 

print("PATH_COL:", PATH_COL)
print("CASEID_COL:", CASEID_COL)
print("SPLIT_COL:", SPLIT_COL)
print("YTRUE_COL:", YTRUE_COL)

if PATH_COL is None:
    raise ValueError(f"Can't find a path column in metadata. columns={list(meta.columns)}")

cfg_path = os.path.join(OUT_DIR, "checkpoint_infer.json")
cfg = json.load(open(cfg_path, "r", encoding="utf-8")) if os.path.exists(cfg_path) else {}
THR = float(cfg.get("threshold", cfg.get("thr", 0.5)))
T   = float(cfg.get("temperature_T", cfg.get("T", 1.0)))
print("THR:", THR, "T:", T)

PCOL = guess_col(meta, ["p_calibrated","p","prob","probability","pred_prob"])
YPRED_COL = guess_col(meta, ["y_pred","pred","yhat","prediction"])

_path2row = {str(p): i for i, p in enumerate(meta[PATH_COL].astype(str).tolist())}

USE_LOOKUP_ONLY = False  # True: 仅查表（只对 rag_meta.csv 里已有图片有效）；False: 真实推理 + RAG

# =========================
# 2) End-to-end inference (方案A): forward_hook 抓 model.backbone.norm_head 输出作为 embedding
#    这与训练时导出 embeddings.npy 的方式一致，无需重训
# =========================
import torch
from PIL import Image
import torchvision.transforms as tvT

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- 读取 checkpoint（优先从 pt 里取 preprocess/postprocess/model_kwargs）----
ckpt_path = os.path.join(OUT_DIR, "checkpoint_infer.pt")
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(f"checkpoint not found: {ckpt_path}")

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

# ---- postprocess: threshold + temperature ----
post = ckpt.get("postprocess", {}) or cfg.get("postprocess", {}) or cfg
THR = float(post.get("cls_threshold", post.get("threshold", post.get("thr", THR))))
T   = float(post.get("temperature_T", post.get("T", T)))
print("THR:", THR, "T:", T)

# ---- preprocess: img_size/mean/std ----
pre = ckpt.get("preprocess", {}) or cfg.get("preprocess", {}) or cfg

def _as_list(v, default):
    # 兼容 v=0.5 / v=[0.5] / v=np.array([0.5])
    if v is None:
        return [float(default)]
    if isinstance(v, (list, tuple)):
        return [float(x) for x in v]
    try:
        import numpy as _np
        if isinstance(v, _np.ndarray):
            return [float(x) for x in v.tolist()]
    except Exception:
        pass
    return [float(v)]

IMG_SIZE = int(pre.get("img_size", pre.get("image_size", cfg.get("img_size", cfg.get("image_size", 512)))))
MEAN = _as_list(pre.get("mean", pre.get("img_mean", cfg.get("mean", cfg.get("img_mean", 0.0)))), default=0.0)
STD  = _as_list(pre.get("std",  pre.get("img_std",  cfg.get("std",  cfg.get("img_std",  1.0)))), default=1.0)

# 单通道：如果 mean/std 给了多通道，只取第一个
MEAN = [MEAN[0]]
STD  = [STD[0]]

preprocess = tvT.Compose([
    tvT.Grayscale(num_output_channels=1),
    tvT.Resize((IMG_SIZE, IMG_SIZE)),
    tvT.ToTensor(),
    tvT.Normalize(mean=MEAN, std=STD),
])

print("IMG_SIZE:", IMG_SIZE, "MEAN:", MEAN, "STD:", STD)

# ---- build model from your model.py (你在文件顶部已经: from model import ConvNeXtV2TinyScratch) ----
model_kwargs = ckpt.get("model_kwargs", {}) or cfg.get("model_kwargs", {})
if not model_kwargs:
    # 兜底：按你训练时常用的参数
    model_kwargs = dict(in_chans=1, n_classes=1, drop_path_rate=0.1, use_seg_guided=False)

print("model_kwargs:", model_kwargs)

infer_model = ConvNeXtV2TinyScratch(**model_kwargs).to(device)

# ---- load state dict（兼容不同键名 + module. 前缀）----
state = (
    ckpt.get("model_state")
    or ckpt.get("model_state_dict")
    or ckpt.get("state_dict")
    or ckpt.get("model")
)
if state is None:
    raise ValueError(f"Checkpoint keys={list(ckpt.keys())}, but no state_dict found.")

if any(k.startswith("module.") for k in state.keys()):
    state = {k.replace("module.", "", 1): v for k, v in state.items()}

miss, unexp = infer_model.load_state_dict(state, strict=False)
if miss:
    print("[Warn] missing keys (show first 10):", miss[:10])
if unexp:
    print("[Warn] unexpected keys (show first 10):", unexp[:10])

infer_model.eval()

import time
import torch.nn.functional as F
from PIL import ImageDraw

def save_seg_overlay_png(
    image_path: str,
    seg_logits: torch.Tensor,
    out_png: str,
    thr: float = 0.5,
    alpha: float = 0.35,
    draw_bbox: bool = True,
):
    """
    Saves a visualization PNG with segmentation overlay and optional bounding box.
    """
    base = Image.open(image_path).convert("RGB")
    w, h = base.size

    prob = torch.sigmoid(seg_logits)  # [1,1,H,W]
    prob = F.interpolate(prob, size=(h, w), mode="bilinear", align_corners=False)[0, 0]
    prob_np = prob.detach().cpu().numpy()  # [h,w], 0~1

    red = (prob_np * 255).clip(0, 255).astype(np.uint8)
    overlay_np = np.zeros((h, w, 3), dtype=np.uint8)
    overlay_np[..., 0] = red  # R通道
    overlay_img = Image.fromarray(overlay_np, mode="RGB")

    blended = Image.blend(base, overlay_img, float(alpha))

    if draw_bbox:
        mask = prob_np >= float(thr)
        if mask.any():
            ys, xs = np.where(mask)
            x1, x2 = int(xs.min()), int(xs.max())
            y1, y2 = int(ys.min()), int(ys.max())
            draw = ImageDraw.Draw(blended)
            lw = max(2, int(min(w, h) * 0.005))
            draw.rectangle([x1, y1, x2, y2], outline=(255, 255, 0), width=lw)

    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    blended.save(out_png)
    return out_png

@torch.no_grad()
def infer_one(image_path: str):
    img = Image.open(image_path).convert("L")
    x = preprocess(img).unsqueeze(0).to(device)

    embed_buf = []
    def _hook(_m, _inp, out):
        embed_buf.append(out.detach())

    handle = infer_model.backbone.norm_head.register_forward_hook(_hook)

    cls_logits, seg_logits = infer_model(x)  # forward return (cls_logits, seg_logits)
    handle.remove()

    if len(embed_buf) == 0:
        raise RuntimeError("Hook did not capture embedding from infer_model.backbone.norm_head")

    emb = embed_buf[0]  
    logits = cls_logits.view(-1).float()

    p = torch.sigmoid(logits / float(T)).item()
    yhat = int(p >= float(THR))

    emb_np = emb.detach().cpu().numpy().astype("float32")
    if emb_np.ndim == 1:
        emb_np = emb_np[None, :]

    # Make sure the embedding dimension matches the library embeddings
    if emb_np.shape[1] != embs.shape[1]:
        raise RuntimeError(f"Embedding dim mismatch: query D={emb_np.shape[1]} vs library D={embs.shape[1]}")

    # Generate and save segmentation overlay PNG for visualization
    base_report_dir = globals().get("REPORT_DIR", os.path.join(OUT_DIR, "reports"))
    overlay_dir = os.path.join(base_report_dir, "overlays")

    stem = Path(image_path).stem
    ts = time.strftime("%Y%m%d_%H%M%S")
    overlay_path = os.path.join(overlay_dir, f"{stem}_{ts}_overlay.png")

    save_seg_overlay_png(
        image_path=image_path,
        seg_logits=seg_logits,
        out_png=overlay_path,
        thr=0.5,       
        alpha=0.35,
        draw_bbox=True
    )
    return emb_np, float(p), int(yhat), overlay_path

# =========================
# 3) Unified interface used by RAG
# =========================
def embed_and_predict(image_path: str):
    if USE_LOOKUP_ONLY:
        # ---- demo: only for images in meta ----
        key = str(image_path)
        if key not in _path2row:
            raise KeyError(f"image_path not found in meta[{PATH_COL}]: {key}")

        i = _path2row[key]
        emb = embs[i:i+1]  # shape (1, D)

        row = meta.iloc[i]
        p = float(row[PCOL]) if (PCOL is not None and PCOL in meta.columns) else 0.5
        yhat = int(row[YPRED_COL]) if (YPRED_COL is not None and YPRED_COL in meta.columns) else int(p >= THR)
        return emb, float(p), int(yhat), None

    # ---- end-to-end: for any new image ----
    return infer_one(image_path)

PATH_COL: full_path
CASEID_COL: image_id
SPLIT_COL: None
YTRUE_COL: y_true
THR: 0.5 T: 0.8587129712104797
THR: 0.37 T: 0.8587129712104797
IMG_SIZE: 576 MEAN: [0.5] STD: [0.25]
model_kwargs: {'in_chans': 1, 'n_classes': 1, 'drop_path_rate': 0.1, 'use_seg_guided': False}


In [4]:
import numpy as np
import faiss

def _l2norm(v: np.ndarray) -> np.ndarray:
    v = v.astype("float32")
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-12)

embs_norm = _l2norm(embs)

N = len(meta)

if SPLIT_COL is None:
    print("[Warn] SPLIT_COL is None -> use full meta as retrieval pool.")
    train_rowids = np.arange(N, dtype=np.int64)
else:
    split = meta[SPLIT_COL].astype(str).str.lower()
    train_rowids = np.flatnonzero(split.isin(["train", "tr"])).astype(np.int64)
    if train_rowids.size == 0:
        print("[Warn] No train rows found in SPLIT_COL -> use full meta as retrieval pool.")
        train_rowids = np.arange(N, dtype=np.int64)

if train_rowids.size == 0:
    print("[Warn] train_rowids is empty -> use full meta as retrieval pool.")
    train_rowids = np.arange(N, dtype=np.int64)

train_embs = embs_norm[train_rowids]
train_index = faiss.IndexFlatIP(train_embs.shape[1])
train_index.add(train_embs)

print("train_index size:", train_index.ntotal, "dim:", train_embs.shape[1])

def image_retrieve_by_vector(q_vec: np.ndarray, topk: int = 5, exclude_case_id=None):
    q = _l2norm(q_vec)
    extra = 10
    scores, idxs = train_index.search(q, topk + extra)

    row_ids = train_rowids[idxs[0]].astype(int)
    hits = meta.iloc[row_ids].copy()
    hits["sim"] = scores[0]

    if exclude_case_id is not None:
        if CASEID_COL is not None and CASEID_COL in hits.columns:
            hits = hits[hits[CASEID_COL].astype(str) != str(exclude_case_id)]
        else:
            from pathlib import Path
            hits = hits[hits[PATH_COL].astype(str).apply(lambda p: Path(p).name) != str(exclude_case_id)]

    return hits.head(topk)

[Warn] SPLIT_COL is None -> use full meta as retrieval pool.
train_index size: 12047 dim: 768


In [5]:
from pathlib import Path

def get_case_id_from_path(image_path: str) -> str:
    return Path(str(image_path)).name 

def shrink_hits_for_llm(hits_df: pd.DataFrame, topk: int):
    out = []
    for _, row in hits_df.head(topk).iterrows():
        item = {
            "case_id": str(row[CASEID_COL]) if (CASEID_COL is not None and CASEID_COL in hits_df.columns)
                      else get_case_id_from_path(row[PATH_COL]),
            "sim": float(row["sim"]) if "sim" in hits_df.columns else None
        }
        for c in ["p_calibrated","p","prob","y_pred","pred","logit"]:
            if c in hits_df.columns:
                v = row[c]
                try:
                    item[c] = int(v) if c in ["y_pred","pred"] else float(v)
                except Exception:
                    item[c] = str(v)
        out.append(item)
    return out

In [6]:
def image_rag_for_image(image_path: str, topk: int = 5):
    emb, p, yhat, overlay_path = embed_and_predict(image_path)

    # exclude self ONLY if the query image exists in the retrieval library
    exclude_id = None
    key = str(image_path)
    if key in _path2row:
        if CASEID_COL is not None and CASEID_COL in meta.columns:
            m = meta[meta[PATH_COL].astype(str) == key]
            exclude_id = str(m.iloc[0][CASEID_COL]) if len(m) > 0 else None
        else:
            exclude_id = None

    hits = image_retrieve_by_vector(emb, topk=topk, exclude_case_id=exclude_id)

    pos_rate_true = None
    if YTRUE_COL is not None and YTRUE_COL in hits.columns:
        try:
            pos_rate_true = float(hits[YTRUE_COL].mean())
        except Exception:
            pos_rate_true = None

    similar_public = shrink_hits_for_llm(hits, topk=topk)
    retrieved_case_ids = [x.get("case_id") for x in similar_public if isinstance(x, dict)]

    return {
        "p_calibrated": float(p),
        "y_pred": int(yhat),
        "threshold": float(THR),
        "temperature_T": float(T),

        "overlay_path": overlay_path,
        "overlay_filename": os.path.basename(overlay_path) if overlay_path else None,

        "similar_cases": similar_public,
        "retrieved_case_ids": retrieved_case_ids,

        "debug": {
            "exclude_case_id": exclude_id,
            "sim_mean": float(hits["sim"].mean()) if "sim" in hits.columns else None,
            "pos_rate_true": pos_rate_true,
        }
    }


In [7]:
import re, json

FORBIDDEN = [
    r"\bleft\b", r"\bright\b", r"laterality",
    r"tension", r"pleural line", r"collapse", r"extent", r"\bsize\b",
    r"apex", r"basal", r"hemithorax",
]

def hallucination_flag(report: dict) -> bool:
    s = str(report.get("diagnostic_report", "")).lower()
    return any(re.search(pat, s) for pat in FORBIDDEN)

In [8]:
import re

def word_count_en(text: str) -> int:
    return len(re.findall(r"\b[\w']+\b", text or ""))

def soft_too_long(text: str, max_words: int = 230, max_chars: int = 1600) -> bool:
    return (word_count_en(text) > max_words) or (len(text) > max_chars)

In [9]:
def fail_safe_report(payload: dict, reason: str) -> dict:
    print("[FAIL_SAFE_REASON]", reason)
    pred = payload.get("prediction", {}) or {}
    p = pred.get("p_calibrated", None)
    thr = pred.get("threshold", None)

    msg = (
        "AI-assisted diagnostic report: Indeterminate for pneumothorax due to a processing safeguard. "
        "The automated system could not generate a reliable concise report. "
        "Recommend radiologist review and clinical correlation."
    )
    if p is not None and thr is not None:
        msg = (
            "AI-assisted diagnostic report: Indeterminate for pneumothorax due to a processing safeguard. "
            f"Model output (p={float(p):.3f}, threshold={float(thr):.3f}) should not be used alone. "
            "Recommend radiologist review and clinical correlation."
        )

    return {"diagnostic_report": msg}

In [10]:
import requests

API_KEY = "sk-6de58e90d64a47ad9163e14cc50067aa"

URL = "https://api.deepseek.com/chat/completions"
MODEL = "deepseek-chat"

def call_llm(system: str, user: str) -> str:
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "stream": False,
        "temperature": 0,
    }
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }
    r = requests.post(URL, json=payload, headers=headers, timeout=180)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

In [11]:
SYSTEM = """You are a consultant radiologist drafting a diagnostic report.

Write a more detailed English diagnostic report. You may use TWO short paragraphs.
Include the following headings (as plain text in-line): Clinical context, Technique, Findings, Impression, Recommendations, Limitations.

Constraints:
- Professional radiology tone, neutral, clinically oriented.
- Do NOT mention: model, calibrated, probability, threshold, retrieval, similar cases, RAG, chunks, IDs, prompts, code, JSON.
- Do NOT invent imaging details not explicitly supported by the input (no laterality, size, extent, tension signs, pleural line, collapse %, etc.).
- It is acceptable to explicitly state that localisation/size/tension assessment is not available from this output.
- Recommendations should be general (radiologist review, clinical correlation, consider further imaging if indicated), no treatment instructions.

Output MUST be valid JSON with exactly one key: "diagnostic_report".
Output MUST start with '{' and end with '}'.
"""

USER_TMPL = """Input JSON:
{payload}

Return ONLY JSON in this exact format:
{{"diagnostic_report":"..."}}
"""

In [12]:
import json

def extract_json_object(s: str) -> str:
    s = s.strip()
    l = s.find("{")
    r = s.rfind("}")
    if l == -1 or r == -1 or r <= l:
        raise ValueError("No JSON object found in LLM output.")
    return s[l:r+1]

def build_user_prompt(payload: dict) -> str:
    payload_json = json.dumps(payload, ensure_ascii=False, indent=2)
    return USER_TMPL.format(payload=payload_json)

def generate_report(payload: dict) -> dict:
    user = build_user_prompt(payload)
    raw = call_llm(SYSTEM, user)

    text = ""

    try:
        obj = extract_json_object(raw)     
        rep = json.loads(obj)
        if isinstance(rep, dict):
            text = str(rep.get("diagnostic_report", "")).strip()
    except Exception:
        pass

    if not text:
        text = str(raw).strip()

    return {"diagnostic_report": text}

In [13]:
def generate_report_safe(payload: dict) -> dict:
    try:
        rep = generate_report(payload)
    except Exception as e:
        return fail_safe_report(payload, f"exception during generation: {e}")

    text = str((rep or {}).get("diagnostic_report", "")).strip()
    if not text:
        return fail_safe_report(payload, "empty diagnostic_report")

    rep = {"diagnostic_report": text}

    if "soft_too_long" in globals() and soft_too_long(text):
        fix_user = (
            "Rewrite the diagnostic report to be concise, single paragraph, professional radiology tone. "
            "Do not invent imaging details. Return JSON only:\n"
            '{"diagnostic_report":"..."}\n\n'
            f"Original:\n{text}"
        )
        try:
            raw2 = call_llm(SYSTEM, fix_user)
            rep2 = json.loads(extract_json_object(raw2))
            t2 = str((rep2 or {}).get("diagnostic_report", "")).strip()
            if t2:
                rep = {"diagnostic_report": t2}
        except Exception:
            pass

    return rep

In [14]:
img = str(meta.iloc[0][PATH_COL])
pack = image_rag_for_image(img, topk=5)

print("exclude_case_id:", pack["debug"]["exclude_case_id"])
print("sim_mean:", pack["debug"]["sim_mean"])
print("similar_cases (top 5):")
print(pd.DataFrame(pack["similar_cases"]).head(5))

exclude_case_id: 1.2.276.0.7230010.3.1.4.8323329.4876.1517875185.207297
sim_mean: 0.07654204219579697
similar_cases (top 5):
                                             case_id       sim  p_calibrated  \
0  1.2.276.0.7230010.3.1.4.8323329.14006.15178752...  0.078253      0.542907   
1  1.2.276.0.7230010.3.1.4.8323329.1038.151787516...  0.077486      0.596445   
2  1.2.276.0.7230010.3.1.4.8323329.4939.151787518...  0.076656      0.530627   
3  1.2.276.0.7230010.3.1.4.8323329.1926.151787517...  0.075422      0.583738   
4  1.2.276.0.7230010.3.1.4.8323329.5038.151787518...  0.074893      0.594341   

   y_pred  
0       1  
1       1  
2       1  
3       1  
4       1  


In [15]:
import os

RAG_DIR = r"C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM"
os.makedirs(RAG_DIR, exist_ok=True)
print("OK:", RAG_DIR)

OK: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM


In [16]:
import os, json

RAG_DIR = r"C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM"
os.makedirs(RAG_DIR, exist_ok=True)

KB_PATH = os.path.join(RAG_DIR, "knowledge.jsonl")

chunks = [
    {"chunk_id":"KB_001","tags":["scope","task"],"text":"Scope: This system provides research-only decision support for pneumothorax binary classification from chest X-ray images. Outputs must be limited to class probability, threshold-based label, and uncertainty/limitations. It must not be used as a standalone diagnostic tool."},
    {"chunk_id":"KB_002","tags":["safety","no_hallucination"],"text":"Restriction: The underlying model is classification-only. Do NOT infer laterality (left/right), size/extent, tension pneumothorax, pleural line, lung collapse percentage, or any localisation/segmentation details unless explicitly provided in the input JSON."},
    {"chunk_id":"KB_003","tags":["output","format"],"text":"Output discipline: Use structured, minimal statements. Prefer 'suggestive of' / 'not suggestive of' / 'indeterminate' rather than definitive diagnostic language. Always include limitations and uncertainty when information is insufficient."},
    {"chunk_id":"KB_004","tags":["evidence","rag"],"text":"RAG rule: Retrieved text chunks provide wording templates and system limitations; retrieved similar images provide model-behavior context only. Do not treat retrieved similar cases as clinical evidence."},
    {"chunk_id":"KB_005","tags":["template","positive"],"text":"Template (positive): The classifier output is suggestive of pneumothorax (binary classification). This result provides no localisation or severity assessment. Human review and clinical correlation are required."},
    {"chunk_id":"KB_006","tags":["template","negative"],"text":"Template (negative): The classifier output is not suggestive of pneumothorax (binary classification). This does not exclude disease in all cases. Human review and clinical correlation remain necessary, especially if clinical concern persists."},
    {"chunk_id":"KB_007","tags":["template","indeterminate"],"text":"Template (indeterminate): The classifier confidence is borderline around the decision threshold. Treat the result as indeterminate; prioritise human review and consider additional information (clinical context and image quality) before relying on the output."},
    {"chunk_id":"KB_008","tags":["confidence","heuristic"],"text":"Confidence heuristic (research): Confidence level may be derived from the absolute distance between calibrated probability p and threshold t (|p−t|). Larger distance suggests higher confidence; small distance suggests borderline uncertainty. This is a heuristic, not a clinical certainty."},
    {"chunk_id":"KB_009","tags":["confidence","high"],"text":"High confidence wording: 'The model output is strongly suggestive of / strongly not suggestive of pneumothorax based on a calibrated probability far from the threshold.' Avoid adding any imaging descriptors."},
    {"chunk_id":"KB_010","tags":["confidence","medium"],"text":"Medium confidence wording: 'The model output is suggestive of / not suggestive of pneumothorax with moderate confidence; interpret with caution and confirm with human review.'"},
    {"chunk_id":"KB_011","tags":["confidence","low","borderline"],"text":"Low confidence wording: 'The model output is borderline near the decision threshold; treat as indeterminate and do not over-interpret.'"},
    {"chunk_id":"KB_012","tags":["calibration","temperature_scaling"],"text":"Calibration note: Probabilities may be temperature-scaled for calibration. Report both the calibrated probability and the temperature parameter if available; calibration improves probabilistic interpretability but does not guarantee correctness per case."},
    {"chunk_id":"KB_013","tags":["threshold","decision"],"text":"Threshold note: The decision threshold is selected from validation (e.g., optimising PR-AUC/F1 or a task-specific utility). Threshold is a policy choice and should be reported explicitly when producing labels."},
    {"chunk_id":"KB_014","tags":["uncertainty","policy"],"text":"Uncertainty policy: When confidence is low or evidence is insufficient, explicitly state uncertainty and recommend human review rather than producing stronger claims."},
    {"chunk_id":"KB_015","tags":["limitations","classification_only"],"text":"Limitation: As a binary classifier, the model cannot provide localisation, laterality, extent, or severity grading. It cannot distinguish visually similar conditions without additional task-specific outputs."},
    {"chunk_id":"KB_016","tags":["limitations","image_quality"],"text":"Limitation: Image quality and acquisition differences (noise, contrast, cropping, rotation, portable views, artifacts) can reduce reliability. If quality indicators are unavailable, note that quality factors may affect performance."},
    {"chunk_id":"KB_017","tags":["limitations","domain_shift"],"text":"Limitation: Domain shift (different hospitals, devices, protocols, preprocessing, demographics) may degrade performance. External/generalisation performance should be referenced at the system level, not assumed per case."},
    {"chunk_id":"KB_018","tags":["limitations","preprocessing"],"text":"Limitation: The model expects a specific preprocessing pipeline (resize, normalisation). Mismatched preprocessing can change outputs; therefore, preprocessing configuration should be versioned and logged."},
    {"chunk_id":"KB_019","tags":["limitations","dataset_bias"],"text":"Limitation: Training data label noise, class imbalance, and sampling choices can bias probabilities and decision thresholds. Report key dataset design decisions in system metadata."},
    {"chunk_id":"KB_020","tags":["actions","generic"],"text":"Action (generic): Use the output for decision support only. Perform human review and correlate with clinical information. Consider further imaging only if clinically indicated (do not prescribe specific interventions or management steps)."},
    {"chunk_id":"KB_021","tags":["actions","borderline"],"text":"Action (borderline): For borderline confidence, prioritise human review, compare with prior studies if available, and avoid making definitive statements based solely on the model output."},
    {"chunk_id":"KB_022","tags":["actions","documentation"],"text":"Documentation: Always include model version, threshold, calibration parameter(s), and retrieval configuration in logs to ensure reproducibility and auditability."},
    {"chunk_id":"KB_023","tags":["similarity","interpretation"],"text":"Similarity interpretation: Nearest-neighbour retrieval reflects feature similarity in embedding space, not ground-truth equivalence. Similarity is useful for model-behaviour inspection and error analysis, not clinical verification."},
    {"chunk_id":"KB_024","tags":["similarity","summary"],"text":"Similarity summary template: 'Among the top-k retrieved similar cases, the observed positive rate is X. This is reported to contextualise model behaviour and does not constitute clinical evidence.'"},
    {"chunk_id":"KB_025","tags":["similarity","conflict"],"text":"Conflict template: 'Retrieved similar cases include label disagreement; this increases uncertainty and supports treating the output as indeterminate or requiring closer human review.'"},
    {"chunk_id":"KB_026","tags":["reporting","minimalism"],"text":"Reporting principle: Prefer minimal faithful reporting over verbose narrative. If the system cannot support a claim, omit it and state limitations instead."},
    {"chunk_id":"KB_027","tags":["reporting","no_localisation"],"text":"No-localisation reminder: Do not mention 'apical/basal', 'pleural line', 'collapse', 'tension', or any positional descriptors. The system does not produce localisation outputs."},
    {"chunk_id":"KB_028","tags":["reporting","label_language"],"text":"Label language: Use 'suggestive of pneumothorax' rather than 'pneumothorax present' when presenting model outputs, unless your evaluation protocol explicitly permits deterministic language."},
    {"chunk_id":"KB_029","tags":["reporting","false_negatives"],"text":"Caution about false negatives: A negative prediction can still occur in true pneumothorax cases, especially under distribution shift or low-quality images. Phrase negatives as 'not suggestive' and emphasise human oversight."},
    {"chunk_id":"KB_030","tags":["reporting","false_positives"],"text":"Caution about false positives: A positive prediction can occur in non-pneumothorax cases; therefore, avoid escalation language and rely on human review and context."},
    {"chunk_id":"KB_031","tags":["metrics","system_level"],"text":"Metrics framing: Performance metrics (AUROC/PR-AUC/sensitivity/specificity) are system-level summaries. Do not translate cohort-level statistics into certainty for an individual case."},
    {"chunk_id":"KB_032","tags":["evaluation","ablation"],"text":"Ablation guidance: Compare (i) no-RAG vs (ii) image-RAG vs (iii) text-RAG vs (iv) dual-RAG using JSON validity, hallucination rate, evidence correctness, and output consistency."},
    {"chunk_id":"KB_033","tags":["evaluation","hallucination_check"],"text":"Hallucination check: Flag outputs containing forbidden localisation/severity terms (e.g., left/right, tension, collapse, extent). Treat such outputs as invalid for a classification-only system."},
    {"chunk_id":"KB_034","tags":["evaluation","consistency"],"text":"Consistency: For research reporting, set generation temperature to 0 (or near 0) and measure stability across repeated runs; instability indicates prompt/constraints are insufficient."},
    {"chunk_id":"KB_035","tags":["engineering","schema"],"text":"Schema rule: Produce strict JSON with fixed keys. Avoid free-form text. This enables automatic validation, safer integration, and clearer evaluation in a graduation project context."},
    {"chunk_id":"KB_036","tags":["engineering","versioning"],"text":"Versioning: Record model architecture name, checkpoint hash, preprocessing parameters, calibration temperature, decision threshold, and retrieval index version for each generated report."},
    {"chunk_id":"KB_037","tags":["engineering","failure_modes"],"text":"Failure-mode reporting: When uncertain, state 'potential factors: acquisition variability, artifacts, domain shift, borderline probability'. Do not speculate about specific radiographic signs."},
    {"chunk_id":"KB_038","tags":["prompting","evidence_binding"],"text":"Evidence binding: Claims in the generated summary should map to (a) prediction fields (p, threshold, temperature) and/or (b) retrieved text chunk IDs. Do not introduce claims without an explicit support source."},
    {"chunk_id":"KB_039","tags":["prompting","retrieval_use"],"text":"Retrieval use: Use text-RAG for allowed wording templates and limitations; use image-RAG to comment on retrieved-case agreement/disagreement with the current prediction for behaviour context."},
    {"chunk_id":"KB_040","tags":["template","confidence_block"],"text":"Confidence block template: 'p_calibrated=..., threshold=..., temperature_T=.... Confidence level is derived from |p−t| and retrieved-case consistency; it is not a clinical certainty.'"},
    {"chunk_id":"KB_041","tags":["template","limitations_block"],"text":"Limitations block template: 'Binary classification only; no localisation or severity; performance may vary with acquisition and domain shift; outputs require human review and clinical correlation.'"},
    {"chunk_id":"KB_042","tags":["template","evidence_block"],"text":"Evidence block template: 'text_chunk_ids=[...]; retrieved_case_ids=[...]. Retrieved cases provide behavioural context only.'"},
    {"chunk_id":"KB_043","tags":["template","uncertainty_block"],"text":"Uncertainty block template: 'Borderline probability and/or conflicting retrieved cases increase uncertainty. Avoid definitive statements and prioritise human confirmation.'"},
    {"chunk_id":"KB_044","tags":["policy","no_treatment"],"text":"Policy: Do not provide treatment or management instructions. Permissible recommendations are limited to 'human review', 'clinical correlation', and 'further imaging if clinically indicated'."},
    {"chunk_id":"KB_045","tags":["policy","no_guideline_claims"],"text":"Policy: Do not claim compliance with a specific external guideline unless that guideline text is explicitly provided in the evidence chunks. Prefer generic, non-prescriptive language."},
    {"chunk_id":"KB_046","tags":["research","thesis_framing"],"text":"Thesis framing: Position the LLM as a constrained natural-language interface for model outputs and retrieval evidence, emphasising auditability, reproducibility, and hallucination control."},
    {"chunk_id":"KB_047","tags":["research","contribution"],"text":"Contribution statement: The dual-RAG design improves interpretability by combining similar-case context (image-RAG) with controlled language templates and limitations (text-RAG), while enforcing strict output constraints."},
    {"chunk_id":"KB_048","tags":["template","indeterminate_trigger"],"text":"Indeterminate trigger (heuristic): If |p−t| is small or retrieved-case labels conflict, classify the narrative impression as 'indeterminate' and emphasise uncertainty."},
    {"chunk_id":"KB_049","tags":["template","positive_brief"],"text":"Brief positive: 'Suggestive of pneumothorax (classification-only). Confirm by human review; no localisation/severity is provided.'"},
    {"chunk_id":"KB_050","tags":["template","negative_brief"],"text":"Brief negative: 'Not suggestive of pneumothorax (classification-only). Human review and clinical correlation are required; false negatives can occur.'"},
    {"chunk_id":"KB_051","tags":["template","indeterminate_brief"],"text":"Brief indeterminate: 'Indeterminate due to borderline confidence and/or conflicting retrieval context. Do not rely on the model output alone.'"},
    {"chunk_id":"KB_052","tags":["confidence","retrieval_consistency"],"text":"Retrieval consistency note: If top-k similar cases largely share the same label as the current prediction, this can be reported as behavioural consistency; otherwise report disagreement as increased uncertainty."},
    {"chunk_id":"KB_053","tags":["logging","audit"],"text":"Audit logging: Store the exact payload (prediction + retrieved chunks + retrieved case IDs) and the final JSON output. This enables deterministic reproduction and debugging."},
    {"chunk_id":"KB_054","tags":["privacy","data_handling"],"text":"Data handling: When reporting retrieved cases, avoid exposing identifiable patient information. Use anonymised case IDs and avoid sensitive metadata in generated summaries."},
    {"chunk_id":"KB_055","tags":["robustness","edge_cases"],"text":"Edge-case wording: If inputs deviate from expected format (unexpected channels, severe resizing, missing normalisation), state that preprocessing mismatch may invalidate outputs."},
    {"chunk_id":"KB_056","tags":["engineering","fail_safe"],"text":"Fail-safe: If JSON validation fails or forbidden terms appear, discard the output and regenerate with stricter constraints or return a minimal 'indeterminate' summary."},
    {"chunk_id":"KB_057","tags":["engineering","separation_of_roles"],"text":"Separation of roles: The classifier produces probabilities; RAG retrieves context; the LLM formats a constrained summary. The LLM must not act as an image interpreter."},
    {"chunk_id":"KB_058","tags":["template","system_note"],"text":"System note template: 'This summary is automatically generated for research and auditing of a pneumothorax classifier. It is not a clinical report and must be reviewed by a qualified human reader.'"},
    {"chunk_id":"KB_059","tags":["template","confidence_mapping"],"text":"Example confidence mapping (heuristic): if |p−t|<0.03 => low; 0.03–0.10 => medium; >0.10 => high. Adjust thresholds to match your calibration and evaluation results."},
    {"chunk_id":"KB_060","tags":["template","retrieval_summary_short"],"text":"Short retrieval summary: 'Top-k similar cases retrieved in embedding space. Positive-rate among retrieved cases is reported for context only.'"}
]

with open(KB_PATH, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print("Saved knowledge to:", KB_PATH)
print("Lines:", len(chunks))

KB_IDS = set([c["chunk_id"] for c in chunks])

Saved knowledge to: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\knowledge.jsonl
Lines: 60


In [17]:
import os, json

KB_PATH = os.path.join(RAG_DIR, "knowledge.jsonl")

kb_chunks = []
with open(KB_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            kb_chunks.append(json.loads(line))

print("KB loaded:", len(kb_chunks))
print("example:", kb_chunks[0])

KB loaded: 60
example: {'chunk_id': 'KB_001', 'tags': ['scope', 'task'], 'text': 'Scope: This system provides research-only decision support for pneumothorax binary classification from chest X-ray images. Outputs must be limited to class probability, threshold-based label, and uncertainty/limitations. It must not be used as a standalone diagnostic tool.'}


In [18]:
import numpy as np

USE_ST = False
text_encoder = None

try:
    from sentence_transformers import SentenceTransformer
    import faiss

    text_encoder = SentenceTransformer("all-MiniLM-L6-v2")
    USE_ST = True
except Exception as e:
    USE_ST = False
    print("[Warn] SentenceTransformer init failed -> fallback TF-IDF. Error:", e)

if USE_ST:
    kb_texts = [c["text"] for c in kb_chunks]
    kb_embs = text_encoder.encode(kb_texts, normalize_embeddings=True).astype("float32")

    text_index = faiss.IndexFlatIP(kb_embs.shape[1])
    text_index.add(kb_embs)

    def text_retrieve(query: str, topk: int = 6):
        q = text_encoder.encode([query], normalize_embeddings=True).astype("float32")
        scores, idxs = text_index.search(q, topk)
        out = []
        for s, i in zip(scores[0], idxs[0]):
            item = dict(kb_chunks[i])
            item["sim"] = float(s)
            out.append(item)
        return out

    print("Text-RAG ready (SentenceTransformer). dim =", kb_embs.shape[1])

else:
    from sklearn.feature_extraction.text import TfidfVectorizer

    kb_texts = [c["text"] for c in kb_chunks]
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    X = vectorizer.fit_transform(kb_texts)

    def text_retrieve(query: str, topk: int = 6):
        q = vectorizer.transform([query])
        scores = (X @ q.T).toarray().ravel()
        idxs = np.argsort(-scores)[:topk]
        out = []
        for i in idxs:
            item = dict(kb_chunks[i])
            item["sim"] = float(scores[i])
            out.append(item)
        return out

    print("Text-RAG ready (TF-IDF).")

Text-RAG ready (SentenceTransformer). dim = 384


In [19]:
def build_text_query(p: float, thr: float) -> str:
    d = abs(p - thr)
    if d < 0.03:
        return "indeterminate borderline classification template; limitations classification-only; uncertainty wording; human review"
    elif d < 0.10:
        return "medium confidence classification template; limitations; calibration temperature scaling; generic actions; evidence binding"
    else:
        return "high confidence classification template; limitations; logging schema; retrieval interpretation"

In [20]:
def build_payload(image_path: str, topk_img: int = 5, topk_text: int = 6) -> dict:
    # 1) image-rag
    pack = image_rag_for_image(image_path, topk=topk_img)

    retrieved_case_ids = pack.get("retrieved_case_ids", [])

    # 3) text-rag
    q = build_text_query(pack["p_calibrated"], pack["threshold"])
    text_hits = text_retrieve(q, topk=topk_text)

    payload = {
        "task": "pneumothorax_binary_classification",
        "prediction": {
            "p_calibrated": float(pack["p_calibrated"]),
            "y_pred": int(pack["y_pred"]),
            "threshold": float(pack["threshold"]),
            "temperature_T": float(pack["temperature_T"]),
        },
        "image_rag": {
            "topk": int(topk_img),
            "retrieved_case_ids": retrieved_case_ids,
            "similar_cases": pack["similar_cases"],
        },
        "text_rag": {
            "query": q,
            "topk": int(topk_text),
            "evidence_chunks": [
                {"chunk_id": t["chunk_id"], "tags": t.get("tags", []), "text": t["text"], "sim": float(t["sim"])}
                for t in text_hits
            ],
        }
    }
    payload["overlay"] = {
        "path": pack.get("overlay_path"),
        "filename": pack.get("overlay_filename"),
    }
    return payload

# quick sanity check: use random image from meta, build payload, print partial output
test_img = meta.iloc[0][PATH_COL]
payload = build_payload(test_img, topk_img=5, topk_text=6)
import json
print(json.dumps(payload, ensure_ascii=False, indent=2)[:1200])

{
  "task": "pneumothorax_binary_classification",
  "prediction": {
    "p_calibrated": 0.039433542639017105,
    "y_pred": 0,
    "threshold": 0.37,
    "temperature_T": 0.8587129712104797
  },
  "image_rag": {
    "topk": 5,
    "retrieved_case_ids": [
      "1.2.276.0.7230010.3.1.4.8323329.14006.1517875249.93352",
      "1.2.276.0.7230010.3.1.4.8323329.1038.1517875166.7134",
      "1.2.276.0.7230010.3.1.4.8323329.4939.1517875185.563513",
      "1.2.276.0.7230010.3.1.4.8323329.1926.1517875170.276641",
      "1.2.276.0.7230010.3.1.4.8323329.5038.1517875186.83236"
    ],
    "similar_cases": [
      {
        "case_id": "1.2.276.0.7230010.3.1.4.8323329.14006.1517875249.93352",
        "sim": 0.07825308293104172,
        "p_calibrated": 0.542906641960144,
        "y_pred": 1
      },
      {
        "case_id": "1.2.276.0.7230010.3.1.4.8323329.1038.1517875166.7134",
        "sim": 0.07748620957136154,
        "p_calibrated": 0.5964450240135193,
        "y_pred": 1
      },
      {
      

In [21]:
def evidence_ok(report: dict, payload: dict) -> bool:
    ev = report.get("evidence", {}) or {}

    txt_ids = ev.get("text_chunk_ids", []) or []
    case_ids = ev.get("retrieved_case_ids", []) or []

    # text ids must come from payload.text_rag if provided
    if "text_rag" in payload:
        allowed_txt = {c["chunk_id"] for c in payload["text_rag"].get("evidence_chunks", [])}
        if not (isinstance(txt_ids, list) and set(txt_ids).issubset(allowed_txt)):
            return False
    else:
        if txt_ids != []:
            return False

    # case ids must come from payload.image_rag if provided
    if "image_rag" in payload:
        allowed_case = set(map(str, payload["image_rag"].get("retrieved_case_ids", []) or []))
        if not (isinstance(case_ids, list) and set(map(str, case_ids)).issubset(allowed_case)):
            return False
    else:
        if case_ids != []:
            return False

    return True

In [22]:
_demo_path = str(meta.iloc[0][PATH_COL])
_demo_payload = build_payload(_demo_path, topk_img=5, topk_text=6)

report = generate_report_safe(_demo_payload)
report

{'diagnostic_report': "Clinical context: This report is generated from an automated analysis of a chest radiograph for the presence of pneumothorax. Technique: The image was processed by a computer-assisted detection system. Findings: The system's analysis indicates a low probability for the presence of a pneumothorax. Specific localisation, size, and assessment for tension are not available from this automated output.\nImpression: No pneumothorax is detected by the automated system. Recommendations: This output requires review by a qualified radiologist. Clinical correlation is essential. Further imaging, such as expiratory views or CT, may be considered if there is a high clinical suspicion. Limitations: This is a binary classification output; it does not provide localisation or quantification. System performance can be affected by image quality, technique, and anatomical variations. The result is not a substitute for a full diagnostic interpretation."}

In [23]:
def payload_llm_only(full: dict) -> dict:
    return {"task": full["task"], "prediction": full["prediction"]}

def payload_image_only(full: dict) -> dict:
    return {"task": full["task"], "prediction": full["prediction"], "image_rag": full["image_rag"]}

def payload_text_only(full: dict) -> dict:
    return {"task": full["task"], "prediction": full["prediction"], "text_rag": full["text_rag"]}

def payload_both(full: dict) -> dict:
    return full

In [24]:
print(meta.columns.tolist())

['image_id', 'full_path', 'new_filename', 'y_true', 'p_calibrated', 'y_pred', 'mask_area_ratio', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 'temperature_T', 'cls_threshold']


In [25]:
print("PATH_COL:", PATH_COL, "CASEID_COL:", CASEID_COL, "SPLIT_COL:", SPLIT_COL, "YTRUE_COL:", YTRUE_COL)

PATH_COL: full_path CASEID_COL: image_id SPLIT_COL: None YTRUE_COL: y_true


In [26]:
import pandas as pd

def run_one(image_path: str):
    full = build_payload(image_path, topk_img=5, topk_text=6)

    groups = {
        "G0_llm_only": payload_llm_only(full),
        "G1_image_rag": payload_image_only(full),
        "G2_text_rag": payload_text_only(full),
        "G3_both": payload_both(full),
    }

    rows = []
    for gname, pld in groups.items():
        base = {
            "group": gname,
            "image_path": image_path,
            "p": pld["prediction"]["p_calibrated"],
            "yhat": pld["prediction"]["y_pred"],
        }
        try:
            rep = generate_report_safe(pld)
            txt = rep.get("diagnostic_report", "")
            rows.append({
                **base,
                "json_ok": isinstance(rep, dict) and "diagnostic_report" in rep,
                "hallucination": hallucination_flag(rep),
                "too_long": soft_too_long(txt),
                "diagnostic_report": txt,
            })
        except Exception as e:
            rows.append({
                **base,
                "json_ok": False,
                "hallucination": False,
                "evidence_ok": False,
                "impression": "",
                "confidence_level": "",
                "error": str(e)
            })
    return rows

sample_paths = meta[PATH_COL].sample(30, random_state=0).astype(str).tolist()

all_rows = []
for pth in sample_paths:
    all_rows += run_one(pth)

df = pd.DataFrame(all_rows)
display(df.head())

summary = df.groupby("group")[["json_ok","hallucination","too_long"]].mean()
display(summary)

out_csv = os.path.join(RAG_DIR, "rag_llm_ablation_results.csv")
df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

,group,image_path,p,yhat,json_ok,hallucination,too_long,diagnostic_report
0,G0_llm_only,C:\Users\Steven\Desktop\Final Project\Datasets...,0.026106,0,True,True,False,Clinical context: The provided information ind...
1,G1_image_rag,C:\Users\Steven\Desktop\Final Project\Datasets...,0.026106,0,True,True,False,Clinical context: The provided information ind...
2,G2_text_rag,C:\Users\Steven\Desktop\Final Project\Datasets...,0.026106,0,True,True,False,Clinical context: Not provided. Technique: Aut...
3,G3_both,C:\Users\Steven\Desktop\Final Project\Datasets...,0.026106,0,True,True,False,Clinical context: The study was performed for ...
4,G0_llm_only,C:\Users\Steven\Desktop\Final Project\Datasets...,0.050103,0,True,True,False,Clinical context: The provided information ind...


,json_ok,hallucination,too_long
group,,,
G0_llm_only,1.0,1.0,0.0
G1_image_rag,1.0,1.0,0.0
G2_text_rag,1.0,1.0,0.0
G3_both,1.0,1.0,0.0


Saved: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\rag_llm_ablation_results.csv


In [27]:
import pandas as pd
import os

out_csv = os.path.join(RAG_DIR, "rag_llm_ablation_results.csv")
df_results = pd.read_csv(out_csv)

path_col = "image_path" if "image_path" in df_results.columns else ("path" if "path" in df_results.columns else None)
print(f"{len(df_results)} records in results file\n")

for i, row in df_results.head(8).iterrows():
    print(f"--- record {i+1} ---")
    if path_col:
        print(f"【image path】: {row.get(path_col)}")
    print(f"[group]: {row.get('group')}")
    print(f"【p_calibrated】: {row.get('p')}, 【yhat】: {row.get('yhat')}")
    print(f"【impression】: {row.get('impression')}")
    print(f"【confidence_level】: {row.get('confidence_level')}")
    print(f"【json_ok】: {row.get('json_ok')}, 【evidence_ok】: {row.get('evidence_ok')}, 【hallucination】: {row.get('hallucination')}")
    if pd.notna(row.get("error", None)):
        print(f"【error】: {row.get('error')}")
    print("-" * 60)

120 records in results file

--- record 1 ---
【image path】: C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_1\Chest X-Ray Images with Pneumothorax Masks\png_images\564_test_0_.png
[group]: G0_llm_only
【p_calibrated】: 0.0261061042547225, 【yhat】: 0
【impression】: None
【confidence_level】: None
【json_ok】: True, 【evidence_ok】: None, 【hallucination】: True
------------------------------------------------------------
--- record 2 ---
【image path】: C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_1\Chest X-Ray Images with Pneumothorax Masks\png_images\564_test_0_.png
[group]: G1_image_rag
【p_calibrated】: 0.0261061042547225, 【yhat】: 0
【impression】: None
【confidence_level】: None
【json_ok】: True, 【evidence_ok】: None, 【hallucination】: True
------------------------------------------------------------
--- record 3 ---
【image path】: C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_1\Chest X-Ray Images with Pneumothorax Masks\png_images\564_test_0_.png
[group]: G2_text_rag
【p_calibrat

In [28]:
from pathlib import Path
import os, time, json

REPORT_DIR = os.path.join(RAG_DIR, "reports")
os.makedirs(REPORT_DIR, exist_ok=True)

def generate_pneumo_report(image_path: str, topk_img: int = 5, topk_text: int = 6, save: bool = True):
    payload = build_payload(image_path, topk_img=topk_img, topk_text=topk_text)
    report = generate_report_safe(payload)
    ov = payload.get("overlay", {}) or {}
    fn = ov.get("filename")
    if fn:
        txt = str(report.get("diagnostic_report", "")).strip()
        suffix = f" See attached overlay image: {fn}."
        if suffix not in txt:
            report["diagnostic_report"] = (txt.rstrip(".") + "." + suffix)
    if ov.get("path"):
        print("🖼️ Overlay:", ov["path"])
    out_path = None
    if save:
        stem = Path(image_path).stem
        ts = time.strftime("%Y%m%d_%H%M%S")
        out_path = os.path.join(REPORT_DIR, f"{stem}_{ts}.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(
                {"image_path": image_path, "payload": payload, "report": report},
                f, ensure_ascii=False, indent=2
            )
        print("✅ Saved:", out_path)

    return report, out_path

In [29]:
img_path = r"C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_2\pneumothorax_normal_balanced_images\00000013_011.png"
rep, saved = generate_pneumo_report(img_path, save=True)
print(rep["diagnostic_report"])

🖼️ Overlay: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\reports\overlays\00000013_011_20260302_232426_overlay.png
✅ Saved: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\reports\00000013_011_20260302_232436.json
Clinical context: Not provided. Technique: A single frontal chest radiograph was analysed using an automated binary classification system for the detection of pneumothorax. Findings: The automated analysis indicates a positive finding for pneumothorax. Specific localisation, size, and assessment for tension are not available from this output.
Impression: Automated analysis suggests the presence of a pneumothorax. Recommendations: This output requires urgent review by a qualified radiologist. Clinical correlation is essential. Further imaging, such as expiratory or lateral decubitus views, or cross-sectional imaging, may be considered if clinically indicated. Limitations: This is a bin

In [30]:
img_path = r"C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_2\pneumothorax_normal_balanced_images\00000013_039.png"
rep, saved = generate_pneumo_report(img_path, save=True)
print(rep["diagnostic_report"])

🖼️ Overlay: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\reports\overlays\00000013_039_20260302_232436_overlay.png
✅ Saved: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\reports\00000013_039_20260302_232446.json
Clinical context: Not provided. Technique: Analysis performed on a single frontal chest radiograph. Findings: The analysis indicates findings consistent with a pneumothorax. Specific localisation, size, and assessment for tension are not available from this output. Impression: Findings are consistent with a pneumothorax. Recommendations: Urgent radiologist review is required. Clinical correlation is essential. Further imaging, such as expiratory or lateral decubitus views, or CT chest, may be considered if clinically indicated to confirm the diagnosis and assess extent. Limitations: This is a binary classification output only. It does not provide localisation, quantification of size, 

In [31]:
img_path = r"C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_2\pneumothorax_normal_balanced_images\00006561_000.png"
rep, saved = generate_pneumo_report(img_path, save=True)
print(rep["diagnostic_report"])

🖼️ Overlay: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\reports\overlays\00006561_000_20260302_232447_overlay.png
✅ Saved: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\reports\00006561_000_20260302_232456.json
Clinical context: Not provided. Technique: Analysis performed on a single frontal chest radiograph. Findings: No definitive evidence of pneumothorax is identified on this assessment. The analysis indicates a low probability for pneumothorax. Specific localization, size, or assessment for tension physiology is not available from this output.

Impression: No pneumothorax detected. Recommendations: This output requires review by a qualified radiologist. Clinical correlation is essential. Further imaging may be considered if clinically indicated. Limitations: This is a binary classification assessment only. It does not provide localization, quantification of size, or evaluation for tensio